# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faja27/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page, tracked for one specific client, on one specific  day (grain: client_hash_id + content_hash_id + report_date).

i developing using a mid-panel month, month=2026-03, and treating the final month (June 2026, the _sample table) as a sealed test month i wont touch until the very end

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: impressions, clicks, avg_position, ctr, sessions — all directly observable on the report date itself

Label/proxy: whether a page visibility declined within the window (a proxy for now, since its based on the current window, not a real future outcome yet)

Context: client_hash_id, content_hash_id (for joining/grouping, not for  modeling), and dim_clients.gsc_data_start / ga4_data_start (to check if a  clients tracking history even covers my window)

Excluded: any FlyRank product decision flags (health_score, priority_score, action_type). these are rule outputs, not observable signals, and using them as features would just make the model copy an existing rule instead of finding real signal

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Query 1 — grain check: no duplicate rows per client+content+date
q1 = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain rows (should be empty):")
print(q1)

# Query 2 — row count and date span for the slice
q2 = con.sql(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("\nRow count and date span:")
print(q2)

# Query 3 — availability: how many rows have usable GSC data
q3 = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as with_gsc
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("\nAvailability check:")
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows (should be empty):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, n]
Index: []

Row count and date span:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability check:
   total_rows   with_gsc
0     9841378  3611061.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cant tell me about clients whose tracking started after march 2026 their rows simply wont exist in this window

More concretely:
only about 37% of rows in this month (3,611,061 out of 9,841,378) have usable GSC data at all the rest are either pre-tracking rows or rows where search visibility just wasnt recorded

i cant treat those missing rows as "zero visibility" without first checking dim_clients.gsc_data_start, or i be mixing "not tracked yet" with "genuinely no traffic" And since im only using one month, i cant yet tell a real decline from normal month-to-month noise that needs a longer window to check persistence

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [v] Every section above is filled — markdown thinking AND the code that backs it
- [v] The notebook runs top to bottom with no errors (Runtime → Run all)
- [v] No client names, URLs, or private queries anywhere
- [v] My claims use careful words: observed, measured, directional, decision-support
- [v] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.